In [1]:
import os
import io
import sys as _sys
# Fix Windows console encoding so Unicode characters print correctly
if hasattr(_sys.stdout, 'reconfigure'):
    _sys.stdout.reconfigure(encoding='utf-8', errors='replace')
import sys
import re
import numpy as np
import scipy.io
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')           # headless safe; swap to 'TkAgg' if you want interactive
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

In [2]:
import os, sys

# In a notebook, use os.getcwd() — Jupyter sets the working directory
# to the folder the notebook is in (i.e. AnalysisExamples/)
notebook_dir = os.getcwd()
sys.path.insert(0, notebook_dir)

# These now work because getSpeechSessionBlocks.py is in the same folder
from getSpeechSessionBlocks import getSpeechSessionBlocks

# neuralDecoder is installed as a package, so this always works regardless
from neuralDecoder.datasets.speechDataset import PHONE_DEF_SIL


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  ← edit these if needed
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR     = 'c:/Users/LENOVO/Koding/Semester 8/TA/speechBCI/data'
SESSION_NAME = 't12.2022.04.28'          # which session to inspect
TRIAL_IDX    = 3                          # which trial (sentence) to zoom into
SPLIT        = 'train'                    # 'train', 'test', or 'competitionHoldOut'
OUT_DIR      = '/eda_figures'   # where to save figures
N_CHANNELS_SHOW = 32                      # how many channels to show in the heatmap

os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=120, bbox_inches='tight')
    print(f"  → saved: {path}")
    plt.close(fig)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1 — RAW .mat INSPECTION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STAGE 1: Raw .mat file")
print("="*60)

mat_path = f'{BASE_DIR}/competitionData/{SPLIT}/{SESSION_NAME}.mat'
dat = scipy.io.loadmat(mat_path)

# ── 1a. Print all top-level keys + shapes ────────────────────────────────────
print(f"\nFile: {mat_path}")
print(f"\n{'Key':<20} {'Shape':<25} {'dtype'}")
print("-" * 55)
for k, v in dat.items():
    if k.startswith('_'):
        continue
    print(f"{k:<20} {str(v.shape):<25} {v.dtype}")

# ── 1b. Show one trial's attributes ──────────────────────────────────────────
n_trials   = dat['sentenceText'].shape[0]
tx1_trial  = dat['tx1'][0, TRIAL_IDX]          # shape: [T, 256]
spk_trial  = dat['spikePow'][0, TRIAL_IDX]     # shape: [T, 256]
sentence   = dat['sentenceText'][TRIAL_IDX].strip()
block_num  = dat['blockIdx'].ravel()[TRIAL_IDX]
T          = tx1_trial.shape[0]

print(f"\n── Trial #{TRIAL_IDX} (block {block_num}) ──────────────────────────")
print(f"  Sentence      : '{sentence}'")
print(f"  Duration      : {T} time bins  ({T * 20:.0f} ms at 50 Hz sampling)")
print(f"  tx1 shape     : {tx1_trial.shape}  (128 ch threshold crossings)")
print(f"  spikePow shape: {spk_trial.shape}  (128 ch spike power)")
print(f"  Feature concat: {np.concatenate([tx1_trial[:, :128], spk_trial[:, :128]], axis=1).shape}  → [T, 256]")
print(f"\nDataset-level stats:")
print(f"  Total trials in this session/split : {n_trials}")
print(f"  Unique blocks (recording sessions) : {sorted(set(dat['blockIdx'].ravel()))}")

# ── 1c. Raw numeric summary for one trial ────────────────────────────────────
features_raw = np.concatenate([tx1_trial[:, :128], spk_trial[:, :128]], axis=1)  # [T, 256]
print(f"\n  Raw feature summary (trial #{TRIAL_IDX}, all 256 channels):")
print(f"    min={features_raw.min():.3f}, max={features_raw.max():.3f}, "
      f"mean={features_raw.mean():.3f}, std={features_raw.std():.3f}")
print(f"    TX1 half  — min={tx1_trial[:,:128].min():.3f}, max={tx1_trial[:,:128].max():.3f}")
print(f"    SpikeP half — min={spk_trial[:,:128].min():.3f}, max={spk_trial[:,:128].max():.3f}")


STAGE 1: Raw .mat file

File: c:/Users/LENOVO/Koding/Semester 8/TA/speechBCI/data/competitionData/train/t12.2022.04.28.mat

Key                  Shape                     dtype
-------------------------------------------------------
sentenceText         (280,)                    <U86
tx1                  (1, 280)                  object
tx2                  (1, 280)                  object
tx3                  (1, 280)                  object
tx4                  (1, 280)                  object
spikePow             (1, 280)                  object
blockIdx             (280, 1)                  uint8

── Trial #3 (block 3) ──────────────────────────
  Sentence      : 'Our experiment's positive outcome was unexpected.'
  Duration      : 461 time bins  (9220 ms at 50 Hz sampling)
  tx1 shape     : (461, 256)  (128 ch threshold crossings)
  spikePow shape: (461, 256)  (128 ch spike power)
  Feature concat: (461, 256)  → [T, 256]

Dataset-level stats:
  Total trials in this session/split 

In [21]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Raw feature heatmaps (tx1 vs spikePow)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[Figure 1] Raw neural features (heatmap) …")
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f'Stage 1: Raw Features — Session {SESSION_NAME}, Trial {TRIAL_IDX}\n'
             f'Sentence: "{sentence}"', fontsize=12, fontweight='bold')

time_ms = np.arange(T) * 20   # 50 Hz → 20 ms per bin

im0 = axes[0].imshow(tx1_trial[:, :N_CHANNELS_SHOW].T, aspect='auto',
                     extent=[time_ms[0], time_ms[-1], N_CHANNELS_SHOW, 0],
                     cmap='hot', vmin=0)
axes[0].set_ylabel('Channel index\n(TX1 — threshold crossings)', fontsize=9)
axes[0].set_title(f'TX1 (threshold crossing counts) — first {N_CHANNELS_SHOW} of 128 channels', fontsize=10)
plt.colorbar(im0, ax=axes[0], label='Count / 20 ms bin')

im1 = axes[1].imshow(spk_trial[:, :N_CHANNELS_SHOW].T, aspect='auto',
                     extent=[time_ms[0], time_ms[-1], N_CHANNELS_SHOW, 0],
                     cmap='viridis', vmin=0)
axes[1].set_ylabel('Channel index\n(Spike Power)', fontsize=9)
axes[1].set_xlabel('Time (ms)', fontsize=10)
axes[1].set_title(f'Spike Power — first {N_CHANNELS_SHOW} of 128 channels', fontsize=10)
plt.colorbar(im1, ax=axes[1], label='Power (a.u.)')

plt.tight_layout()
save(fig, '1_raw_heatmap.png')



[Figure 1] Raw neural features (heatmap) …
  → saved: AnalysisExamples/eda_figures\1_raw_heatmap.png


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Single channel time series (raw)
# ═══════════════════════════════════════════════════════════════════════════════
print("[Figure 2] Single-channel time series (raw) …")
CHANNELS_TO_PLOT = [0, 10, 50, 100]
fig, axes = plt.subplots(len(CHANNELS_TO_PLOT), 2, figsize=(14, 8), sharex=True)
fig.suptitle(f'Stage 1: Channel Time Series — "{sentence}"', fontsize=12, fontweight='bold')

for row, ch in enumerate(CHANNELS_TO_PLOT):
    axes[row, 0].plot(time_ms, tx1_trial[:, ch], color='tomato', linewidth=0.8)
    axes[row, 0].set_ylabel(f'Ch {ch}', fontsize=8)
    if row == 0:
        axes[row, 0].set_title('TX1 (threshold crossings)', fontsize=10)

    axes[row, 1].plot(time_ms, spk_trial[:, ch], color='steelblue', linewidth=0.8)
    if row == 0:
        axes[row, 1].set_title('Spike Power', fontsize=10)

axes[-1, 0].set_xlabel('Time (ms)')
axes[-1, 1].set_xlabel('Time (ms)')
plt.tight_layout()
save(fig, '2_raw_timeseries.png')

[Figure 2] Single-channel time series (raw) …
  → saved: AnalysisExamples/eda_figures\2_raw_timeseries.png


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Distribution of raw features (all trials, this session)
# ═══════════════════════════════════════════════════════════════════════════════
print("[Figure 3] Feature value distributions (all trials) …")
all_tx1, all_spk = [], []
for i in range(min(n_trials, 50)):   # sample first 50 trials for speed
    all_tx1.append(dat['tx1'][0, i][:, :128].ravel())
    all_spk.append(dat['spikePow'][0, i][:, :128].ravel())
all_tx1 = np.concatenate(all_tx1)
all_spk = np.concatenate(all_spk)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Stage 1: Value Distribution — Session {SESSION_NAME} (first 50 trials)', fontsize=11)

axes[0].hist(all_tx1, bins=80, color='tomato', alpha=0.8, log=True)
axes[0].set_xlabel('TX1 value')
axes[0].set_ylabel('Count (log)')
axes[0].set_title('TX1 (threshold crossings)')
axes[0].axvline(np.mean(all_tx1), color='black', linestyle='--', label=f'mean={np.mean(all_tx1):.2f}')
axes[0].legend(fontsize=8)

axes[1].hist(all_spk, bins=80, color='steelblue', alpha=0.8, log=True)
axes[1].set_xlabel('Spike Power value')
axes[1].set_ylabel('Count (log)')
axes[1].set_title('Spike Power')
axes[1].axvline(np.mean(all_spk), color='black', linestyle='--', label=f'mean={np.mean(all_spk):.2f}')
axes[1].legend(fontsize=8)

plt.tight_layout()
save(fig, '3_raw_distributions.png')

[Figure 3] Feature value distributions (all trials) …
  → saved: AnalysisExamples/eda_figures\3_raw_distributions.png


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Trial duration distribution
# ═══════════════════════════════════════════════════════════════════════════════
print("[Figure 4] Trial duration distribution …")
durations_ms = [dat['tx1'][0, i].shape[0] * 20 for i in range(n_trials)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations_ms, bins=30, color='mediumpurple', alpha=0.85, edgecolor='white')
ax.set_xlabel('Trial duration (ms)', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title(f'Stage 1: Trial Duration Distribution — Session {SESSION_NAME}', fontsize=11)
ax.axvline(np.mean(durations_ms), color='black', linestyle='--',
           label=f'mean = {np.mean(durations_ms):.0f} ms')
ax.legend()
plt.tight_layout()
save(fig, '4_trial_durations.png')

[Figure 4] Trial duration distribution …
  → saved: AnalysisExamples/eda_figures\4_trial_durations.png


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 — TFRecord INSPECTION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STAGE 2: TFRecord file (processed, normalized)")
print("="*60)

tfr_path = f'{BASE_DIR}/derived/tfRecords/{SESSION_NAME}/{SPLIT}/chunk_0.tfrecord'
if not os.path.exists(tfr_path):
    print(f"  [!] TFRecord not found at {tfr_path}")
    print("  Run rnn_step1_makeTFRecords.py first, then re-run this script.")
    sys.exit(0)


STAGE 2: TFRecord file (processed, normalized)


In [12]:
# ── parse TFRecord ────────────────────────────────────────────────────────────
feature_desc = {
    'inputFeatures':   tf.io.VarLenFeature(tf.float32),
    'seqClassIDs':     tf.io.VarLenFeature(tf.int64),
    'nTimeSteps':      tf.io.VarLenFeature(tf.int64),
    'nSeqElements':    tf.io.VarLenFeature(tf.int64),
    'ceMask':          tf.io.VarLenFeature(tf.float32),
    'newClassSignal':  tf.io.VarLenFeature(tf.float32),
    'classLabelsOneHot': tf.io.VarLenFeature(tf.float32),
    'transcription':   tf.io.VarLenFeature(tf.int64),
}

records = []
for raw in tf.data.TFRecordDataset(tfr_path):
    parsed = tf.io.parse_single_example(raw, feature_desc)
    nT  = int(tf.sparse.to_dense(parsed['nTimeSteps']).numpy()[0])
    nPh = int(tf.sparse.to_dense(parsed['nSeqElements']).numpy()[0])
    feats = tf.sparse.to_dense(parsed['inputFeatures']).numpy().reshape(nT, 256)
    seqIDs = tf.sparse.to_dense(parsed['seqClassIDs']).numpy()[:nPh]  # shape: [nPh]
    transcription_ascii = tf.sparse.to_dense(parsed['transcription']).numpy()
    text = ''.join([chr(c) for c in transcription_ascii if c > 0])
    records.append({'nTimeSteps': nT, 'nSeqElements': nPh,
                    'inputFeatures': feats, 'seqClassIDs': seqIDs, 'text': text})

print(f"\nTFRecord: {tfr_path}")
print(f"  Total trials stored: {len(records)}")


TFRecord: c:/Users/LENOVO/Koding/Semester 8/TA/speechBCI/data/derived/tfRecords/t12.2022.04.28/train/chunk_0.tfrecord
  Total trials stored: 280


In [13]:
# ── Print per-field summary ───────────────────────────────────────────────────
rec = records[TRIAL_IDX]
print(f"\n── Trial #{TRIAL_IDX} ─────────────────────────────────────────────")
print(f"  text          : '{rec['text']}'")
print(f"  nTimeSteps    : {rec['nTimeSteps']}  ← actual neural duration in bins")
print(f"  nSeqElements  : {rec['nSeqElements']}  ← number of phonemes in this sentence")
print(f"  inputFeatures : shape={rec['inputFeatures'].shape}, dtype=float32")
print(f"    min={rec['inputFeatures'].min():.3f}, max={rec['inputFeatures'].max():.3f}, "
      f"mean={rec['inputFeatures'].mean():.4f}, std={rec['inputFeatures'].std():.3f}")
print(f"  seqClassIDs   : {rec['seqClassIDs']}")
phoneme_names = [PHONE_DEF_SIL[i - 1] for i in rec['seqClassIDs'] if i > 0]
print(f"  phoneme names : {phoneme_names}")
print(f"\nFields in every TFRecord entry:")
print(f"  {'Field':<22} {'Explanation'}")
print(f"  {'-'*55}")
fields = [
    ('inputFeatures',    '[T × 256] block-normalized neural features (TX1 + spikePow)'),
    ('nTimeSteps',       'actual T for this trial (sequence length in time bins)'),
    ('seqClassIDs',      'phoneme ID sequence, padded to 500  (1-indexed, 0=padding)'),
    ('nSeqElements',     'number of real phonemes in seqClassIDs'),
    ('ceMask',           '[T] binary mask: 1 during speech, 0 for padding'),
    ('newClassSignal',   '[T] 1 at phoneme boundaries (used only for CE loss)'),
    ('classLabelsOneHot','[T × 31] one-hot phoneme labels per time bin (CE loss only)'),
    ('transcription',    '[500] ASCII codes of the original sentence text'),
]
for name, desc in fields:
    print(f"  {name:<22} {desc}")


── Trial #3 ─────────────────────────────────────────────
  text          : 'our experiment's positive outcome was unexpected'
  nTimeSteps    : 461  ← actual neural duration in bins
  nSeqElements  : 46  ← number of phonemes in this sentence
  inputFeatures : shape=(461, 256), dtype=float32
    min=-2.722, max=22.882, mean=0.0536, std=1.025
  seqClassIDs   : [ 5 12 40 17 20 29 27 11 28  3 22  3 23 31 29 40 27  1 38  3 31 17 35 40
  5 31 20  3 22 40 36  1 38 40  3 23 17 20 29 27 11 20 31 17  9 40]
  phoneme names : ['AW', 'ER', 'SIL', 'IH', 'K', 'S', 'P', 'EH', 'R', 'AH', 'M', 'AH', 'N', 'T', 'S', 'SIL', 'P', 'AA', 'Z', 'AH', 'T', 'IH', 'V', 'SIL', 'AW', 'T', 'K', 'AH', 'M', 'SIL', 'W', 'AA', 'Z', 'SIL', 'AH', 'N', 'IH', 'K', 'S', 'P', 'EH', 'K', 'T', 'IH', 'D', 'SIL']

Fields in every TFRecord entry:
  Field                  Explanation
  -------------------------------------------------------
  inputFeatures          [T × 256] block-normalized neural features (TX1 + spikePow)
  nTim

In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — Processed feature heatmap (TFRecord)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[Figure 5] Processed (normalized) feature heatmap …")
feats = rec['inputFeatures']   # [T, 256]
T_rec = feats.shape[0]
time_ms_rec = np.arange(T_rec) * 20

fig, axes = plt.subplots(3, 1, figsize=(14, 9))
fig.suptitle(f'Stage 2: Processed Features (TFRecord) — Trial {TRIAL_IDX}\n'
             f'Sentence: "{rec["text"]}"  →  {rec["nSeqElements"]} phonemes: {phoneme_names}',
             fontsize=11, fontweight='bold')

# TX1 half (channels 0-127)
im0 = axes[0].imshow(feats[:, :N_CHANNELS_SHOW].T, aspect='auto',
                     extent=[time_ms_rec[0], time_ms_rec[-1], N_CHANNELS_SHOW, 0],
                     cmap='RdBu_r', vmin=-3, vmax=3)
axes[0].set_title(f'TX1 channels (first {N_CHANNELS_SHOW}/128) — AFTER block normalization', fontsize=10)
axes[0].set_ylabel('Channel')
plt.colorbar(im0, ax=axes[0], label='z-score')

# SpikeP half (channels 128-255)
im1 = axes[1].imshow(feats[:, 128:128+N_CHANNELS_SHOW].T, aspect='auto',
                     extent=[time_ms_rec[0], time_ms_rec[-1], N_CHANNELS_SHOW, 0],
                     cmap='RdBu_r', vmin=-3, vmax=3)
axes[1].set_title(f'Spike Power channels (first {N_CHANNELS_SHOW}/128) — AFTER block normalization', fontsize=10)
axes[1].set_ylabel('Channel')
plt.colorbar(im1, ax=axes[1], label='z-score')

# Mean activity across all 256 channels
mean_activity = feats.mean(axis=1)
axes[2].plot(time_ms_rec, mean_activity, color='darkgreen', linewidth=1.0)
axes[2].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Mean z-score\n(all 256 ch)')
axes[2].set_title('Mean activity across all 256 channels', fontsize=10)

plt.tight_layout()
save(fig, '5_processed_heatmap.png')


[Figure 5] Processed (normalized) feature heatmap …
  → saved: AnalysisExamples/eda_figures\5_processed_heatmap.png


In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 6 — Before vs After normalization comparison (one channel)
# ═══════════════════════════════════════════════════════════════════════════════
print("[Figure 6] Before vs After normalization (channel 5) …")
CH = 5
raw_ch  = dat['tx1'][0, TRIAL_IDX][:, CH]    # raw single channel
norm_ch = rec['inputFeatures'][:, CH]          # same channel after normalization

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle(f'Before vs After Block Normalization — TX1 Channel {CH}\n"{rec["text"]}"', fontsize=11)

axes[0].plot(time_ms, raw_ch, color='tomato', linewidth=0.9)
axes[0].set_ylabel('Raw count / 20ms bin')
axes[0].set_title(f'BEFORE: raw tx1  (mean={raw_ch.mean():.2f}, std={raw_ch.std():.2f})')

axes[1].plot(time_ms_rec, norm_ch, color='steelblue', linewidth=0.9)
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_ylabel('z-score')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title(f'AFTER: normalized  (mean={norm_ch.mean():.4f}, std={norm_ch.std():.2f})')

plt.tight_layout()
save(fig, '6_before_vs_after_norm.png')

[Figure 6] Before vs After normalization (channel 5) …
  → saved: AnalysisExamples/eda_figures\6_before_vs_after_norm.png


In [18]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 7 — Phoneme alphabet + class ID mapping
# ═══════════════════════════════════════════════════════════════════════════════
print("[Figure 7] Phoneme alphabet (all 41 classes) …")
fig, ax = plt.subplots(figsize=(12, 3))
fig.suptitle('Phoneme Alphabet (PHONE_DEF_SIL) — the 41 target classes', fontsize=11)

n_phones = len(PHONE_DEF_SIL)
ax.imshow(np.arange(n_phones).reshape(1, -1), cmap='tab20', aspect='auto')
for i, p in enumerate(PHONE_DEF_SIL):
    ax.text(i, 0, f'{p}\n{i+1}', ha='center', va='center', fontsize=7,
            color='white' if i % 2 == 0 else 'black', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel(f'ID 1 … {n_phones}   (0 = padding/blank for CTC)', fontsize=9)
plt.tight_layout()
save(fig, '7_phoneme_alphabet.png')

[Figure 7] Phoneme alphabet (all 41 classes) …
  → saved: AnalysisExamples/eda_figures\7_phoneme_alphabet.png


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 8 — Dataset-level statistics (all trials in TFRecord)
# ═══════════════════════════════════════════════════════════════════════════════
print("[Figure 8] Dataset-level stats (all trials in TFRecord) …")
all_T  = [r['nTimeSteps'] for r in records]
all_Ph = [r['nSeqElements'] for r in records]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Stage 2: Dataset-level Statistics — {SESSION_NAME}/{SPLIT}', fontsize=11)

axes[0].hist(all_T, bins=25, color='mediumpurple', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('nTimeSteps (time bins)')
axes[0].set_ylabel('Trials')
axes[0].set_title(f'Neural sequence length\nmean={np.mean(all_T):.0f} bins = {np.mean(all_T)*20:.0f} ms')

axes[1].hist(all_Ph, bins=20, color='darkorange', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('nSeqElements (# phonemes)')
axes[1].set_ylabel('Trials')
axes[1].set_title(f'Phoneme sequence length\nmean={np.mean(all_Ph):.1f} phonemes/sentence')

# Phoneme ID frequency across all trials
all_ids = np.concatenate([r['seqClassIDs'] for r in records])
unique, counts = np.unique(all_ids, return_counts=True)
phone_labels = [PHONE_DEF_SIL[u - 1] if u > 0 else 'PAD' for u in unique]
axes[2].bar(phone_labels, counts, color='teal', alpha=0.85)
axes[2].set_xlabel('Phoneme')
axes[2].set_ylabel('Count')
axes[2].set_title('Phoneme frequency')
axes[2].tick_params(axis='x', rotation=90, labelsize=6)

plt.tight_layout()
save(fig, '8_dataset_stats.png')

[Figure 8] Dataset-level stats (all trials in TFRecord) …


Text(0.5, 1.0, 'Phoneme sequence length\nmean=44.7 phonemes/sentence')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("DONE. All figures saved to:", OUT_DIR)
print("="*60)
print("\nFigures summary:")
figs = [
    ('1_raw_heatmap.png',          'Raw TX1 + SpikeP heatmaps side-by-side for one trial'),
    ('2_raw_timeseries.png',       'Individual channel time series (raw) for 4 channels'),
    ('3_raw_distributions.png',    'Value distribution of TX1 and SpikeP across 50 trials'),
    ('4_trial_durations.png',      'How long each trial (sentence) is in ms'),
    ('5_processed_heatmap.png',    'Normalized feature heatmaps + mean activity after TFRecord'),
    ('6_before_vs_after_norm.png', 'One channel before and after block normalization'),
    ('7_phoneme_alphabet.png',     'All 41 phoneme classes with their integer IDs'),
    ('8_dataset_stats.png',        'Trial length, phoneme count, and phoneme frequency across all trials'),
]
for fname, desc in figs:
    print(f"  {fname:<35} {desc}")

  → saved: AnalysisExamples/eda_figures\8_dataset_stats.png
